# 00 — Teste de fumaça do acesso aos dados (Fase 0, passo 0.5)

Caderno temporário, **não faz parte da entrega**. Verifica se as fontes podem ser baixadas por código antes de investir na extração completa. Mostra apenas nomes de colunas e contagens — nenhum registro individual é impresso.

**Localização do repositório.** Garante que o caderno rode a partir de `notebooks/`, inclusive quando aberto isolado (ex.: Google Colab), caso em que o repositório público é clonado. Todos os caminhos seguintes são relativos à raiz.

In [2]:
# Localiza a raiz do repositório (pasta com requirements.txt e src/appd).
# Se o caderno foi aberto isolado (ex.: Google Colab), clona o repositório
# público na pasta atual. Depois disso, tudo usa caminhos relativos à raiz.
import os
import subprocess
from pathlib import Path

URL_REPOSITORIO = "https://github.com/bsousa7/APPD_26.git"


def localizar_raiz():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "requirements.txt").exists() and (pasta / "src" / "appd").exists():
            return pasta
    return None


if localizar_raiz() is None:
    copia = Path("APPD_26")
    if (copia / ".git").exists():
        subprocess.run(["git", "-C", str(copia), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(copia)], check=True)
    os.chdir(copia)
    if localizar_raiz() is None:
        raise RuntimeError(
            f"A cópia em {Path.cwd()} não tem requirements.txt e src/appd. "
            "Apague a pasta APPD_26 e execute esta célula de novo."
        )
os.chdir(localizar_raiz() / "notebooks")
versao = subprocess.run(["git", "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip()
print(f"Pasta de trabalho: {Path.cwd().parent.name}/notebooks | versão do repositório: {versao}")

Pasta de trabalho: APPD_26/notebooks | versão do repositório: b5a704b 2026-11-22


## 1. Ambiente
Instala dependências, define caminhos e aponta o cache do PySUS para `data/raw/pysus` (ignorado pelo Git).

In [3]:
# Escolhe o arquivo de requisitos conforme o ambiente:
# - Google Colab: requirements-colab.txt (só o que o Colab não traz, sem
#   substituir pandas, ipykernel etc., que o próprio Colab exige);
# - ambiente local: requirements.txt (todas as bibliotecas, versões fixadas).
# O caminho é calculado a partir de notebooks/ e passado já resolvido ao %pip.
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
nome = "requirements-colab.txt" if NO_COLAB else "requirements.txt"
ARQ_REQUISITOS = (Path.cwd().parent / nome).resolve()
if not ARQ_REQUISITOS.exists():
    raise FileNotFoundError(
        f"{nome} não encontrado a partir de {Path.cwd().name}/. "
        "Execute antes a célula 'Localização do repositório' (seção 1)."
    )
print("Instalando", nome)
%pip install -q -r {ARQ_REQUISITOS}

# Se o pip trocou a versão de uma biblioteca que já estava carregada (comum no
# Colab, que pré-carrega numpy e pandas), a sessão precisa ser reiniciada.
from importlib.metadata import version as versao_instalada

trocadas = [
    f"{nome} {sys.modules[nome].__version__} -> {versao_instalada(nome)}"
    for nome in ["numpy", "pandas", "scipy"]
    if nome in sys.modules and sys.modules[nome].__version__ != versao_instalada(nome)
]
if trocadas:
    raise RuntimeError(
        "Bibliotecas atualizadas: " + "; ".join(trocadas) + ". Reinicie a sessão "
        "(Ambiente de execução > Reiniciar sessão) e execute TUDO de novo, desde a "
        "primeira célula de código. Na segunda execução esta célula passa direto."
    )
print("Dependências prontas.")

Instalando requirements-colab.txt
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 2.8.2 which is incompatible.
Dependências prontas.


In [4]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
if not (RAIZ / "src" / "appd").exists():
    raise RuntimeError(
        "Pasta de trabalho errada (comum após reiniciar a sessão do Colab). "
        "Execute o caderno desde a primeira célula de código (localização do repositório)."
    )
sys.path.insert(0, str(RAIZ / "src"))

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
from importlib.metadata import version
print("Python", sys.version.split()[0], "|", ", ".join(
    f"{p} {version(p)}" for p in ["pandas", "numpy", "scipy", "statsmodels",
                                  "scikit-learn", "linearmodels", "differences", "pysus"]))

import pysus
from appd import download as dl
from appd.agregacao import TP_UNID_POLO

dl.configurar_cache(RAIZ)
print("PySUS", pysus.__version__)

Python 3.13.18 | pandas 2.2.3, numpy 2.4.6, scipy 1.16.3, statsmodels 0.15.0, scikit-learn 1.6.1, linearmodels 7.1, differences 0.3.0, pysus 2.12.2
PySUS 2.12.2 -- welcome!
Data cache: /root/pysus
Change it with: pysus.set_cache('/your/path')
Browse datasets with: pysus.info()

PySUS 2.12.2


O cache do PySUS passa a ficar dentro de `data/raw/`, que é esvaziado na entrega.

## 2. CNES-ST: em que ano o tipo 74 aparece?
**Por quê:** confirmar que o campo `TP_UNID` existe e descobrir a primeira competência com estabelecimentos do tipo 74 (Polo Academia da Saúde). A hipótese inicial era que o tipo só existisse a partir da Portaria SAS/MS nº 24/2014; o resultado abaixo (PE com polos tipo 74 já em 2012) a refuta.

Lê só `CNES`, `CODUFMUN` e `TP_UNID`: o arquivo completo traz `CPF_CNPJ`, que não deve ser carregado (Política de IA, art. 4º, VI). Usa DF e PE (estado com muitos polos) em dezembro de 2012 a 2016.

In [5]:
COLS_ST = ["CNES", "CODUFMUN", "TP_UNID"]
contagem = {}
for uf in ["DF", "PE"]:
    for ano in range(2012, 2017):
        st = dl.baixar_datasus("cnes", uf, ano, 12, grupo="ST", colunas=COLS_ST)
        assert set(COLS_ST) <= set(st.columns)
        contagem[(uf, ano)] = int((st["TP_UNID"].str.strip().str.zfill(2) == TP_UNID_POLO).sum())
pd.Series(contagem, name="estabelecimentos tipo 74").unstack(0)

STDF1212.parquet: 100%|██████████| 355k/355k [00:00<00:00, 1.18MB/s]
STDF1312.parquet: 100%|██████████| 369k/369k [00:00<00:00, 1.20MB/s]
STDF1412.parquet: 100%|██████████| 378k/378k [00:00<00:00, 1.09MB/s]
STDF1512.parquet: 100%|██████████| 340k/340k [00:00<00:00, 1.14MB/s]
STDF1612.parquet: 100%|██████████| 319k/319k [00:00<00:00, 1.12MB/s]
STPE1212.parquet: 100%|██████████| 409k/409k [00:00<00:00, 1.23MB/s]
STPE1312.parquet: 100%|██████████| 430k/430k [00:00<00:00, 1.28MB/s]
STPE1412.parquet: 100%|██████████| 451k/451k [00:00<00:00, 1.25MB/s]
STPE1512.parquet: 100%|██████████| 433k/433k [00:00<00:00, 1.29MB/s]
STPE1612.parquet: 100%|██████████| 446k/446k [00:00<00:00, 1.33MB/s]


,DF,PE
2012,0,137
2013,0,175
2014,0,201
2015,0,207
2016,0,222


**Resultado:** [PREENCHER após executar: primeiro ano com polos tipo 74 em cada UF]

## 2b. Varredura nacional do tipo 74, dez/2008 a dez/2016
**Por quê:** a seção 2 mostrou 0 polos tipo 74 no DF em todos os anos, mas 137 em PE já em dez/2012. Logo, o tipo 74 existe antes de 2014 e o resultado do DF não vale para o país. Esta célula conta, para todas as UFs, os estabelecimentos tipo 74 e os municípios com ao menos um, para definir com evidência o ano da primeira coorte.

Lê só `CNES`, `CODUFMUN` e `TP_UNID` (sem `CPF_CNPJ`). São 27 UFs × 9 anos = 243 arquivos pequenos; os brutos são apagados após a leitura.

In [ ]:
UFS = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA",
       "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]
linhas = []
for ano in range(2008, 2017):
    for uf in UFS:
        st = dl.baixar_datasus("cnes", uf, ano, 12, grupo="ST", colunas=COLS_ST)
        polos = st.loc[st["TP_UNID"].str.strip().str.zfill(2) == TP_UNID_POLO]
        linhas.append({"ano": ano, "uf": uf, "polos": polos["CNES"].nunique(),
                       "municipios": polos["CODUFMUN"].nunique()})
varredura = pd.DataFrame(linhas)
assert not varredura.duplicated(["ano", "uf"]).any() and len(varredura) == 9 * len(UFS)

print("Brasil, dezembro de cada ano:")
display(varredura.groupby("ano")[["polos", "municipios"]].sum().T)
primeiro = varredura.loc[varredura["polos"] > 0].groupby("uf")["ano"].min()
print("Primeiro ano com tipo 74, por UF (UFs ausentes não tiveram nenhum até 2016):")
primeiro.sort_values().to_frame("primeiro ano").T

**Resultado:** [PREENCHER após executar: primeiro ano com tipo 74 no país; se há salto em algum ano (por exemplo, na criação ou na redefinição do tipo); quais UFs começam mais cedo]

## 3. SIH-RD, DF, janeiro de 2024
**Por quê:** confirmar os campos `MUNIC_RES`, `DIAG_PRINC` e `ANO_CMPT` e a disponibilidade de 2024.

In [6]:
rd = dl.baixar_datasus("sih", "DF", 2024, 1, grupo="RD")
print("Colunas:", list(rd.columns)[:40], "...")
print("Campos necessários presentes:", {c: c in rd.columns for c in ["MUNIC_RES", "DIAG_PRINC", "ANO_CMPT", "IDADE", "COD_IDADE"]})
print("Linhas:", len(rd))

RDDF2401.parquet: 100%|██████████| 1.20M/1.20M [00:00<00:00, 2.31MB/s]


Colunas: ['UF_ZI', 'ANO_CMPT', 'MES_CMPT', 'ESPEC', 'CGC_HOSP', 'N_AIH', 'IDENT', 'CEP', 'MUNIC_RES', 'NASC', 'SEXO', 'UTI_MES_IN', 'UTI_MES_AN', 'UTI_MES_AL', 'UTI_MES_TO', 'MARCA_UTI', 'UTI_INT_IN', 'UTI_INT_AN', 'UTI_INT_AL', 'UTI_INT_TO', 'DIAR_ACOM', 'QT_DIARIAS', 'PROC_SOLIC', 'PROC_REA', 'VAL_SH', 'VAL_SP', 'VAL_SADT', 'VAL_RN', 'VAL_ACOMP', 'VAL_ORTP', 'VAL_SANGUE', 'VAL_SADTSR', 'VAL_TRANSP', 'VAL_OBSANG', 'VAL_PED1AC', 'VAL_TOT', 'VAL_UTI', 'US_TOT', 'DT_INTER', 'DT_SAIDA'] ...
Campos necessários presentes: {'MUNIC_RES': True, 'DIAG_PRINC': True, 'ANO_CMPT': True, 'IDADE': True, 'COD_IDADE': True}
Linhas: 19950


**Resultado:** [PREENCHER após executar]

## 4. IBGE: lista de municípios e população (SIDRA)
**Por quê:** confirmar a API de localidades e o número da tabela de estimativas populacionais (provavelmente 6579, variável 9324).

In [7]:
mun = dl.municipios_ibge()
print("Municípios:", len(mun), "| UFs:", mun["uf"].nunique())
pop = dl.tabela_sidra("6579", "9324", "2015")
print("Colunas SIDRA:", list(pop.columns))
print("Linhas:", len(pop))

/usr/local/lib/python3.13/dist-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.7.0)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(


Municípios: 5571 | UFs: 27
Colunas SIDRA: ['Nível Territorial (Código)', 'Nível Territorial', 'Unidade de Medida (Código)', 'Unidade de Medida', 'Valor', 'Município (Código)', 'Município', 'Ano (Código)', 'Ano', 'Variável (Código)', 'Variável']
Linhas: 5571


**Resultado:** [PREENCHER após executar]

## 5. Registro de acessos e versões

In [8]:
display(dl.registro_acessos())
%pip show pysus differences linearmodels

,fonte,parametros,origem_efetiva,linhas,acesso_em
0,CNES-ST,"{'uf': 'DF', 'ano': 2012, 'mes': 12}",FTP DATASUS,5470,2026-11-22T10:09:28
1,CNES-ST,"{'uf': 'DF', 'ano': 2013, 'mes': 12}",FTP DATASUS,5830,2026-11-22T10:09:31
2,CNES-ST,"{'uf': 'DF', 'ano': 2014, 'mes': 12}",FTP DATASUS,6050,2026-11-22T10:09:33
3,CNES-ST,"{'uf': 'DF', 'ano': 2015, 'mes': 12}",FTP DATASUS,5428,2026-11-22T10:09:36
4,CNES-ST,"{'uf': 'DF', 'ano': 2016, 'mes': 12}",FTP DATASUS,4906,2026-11-22T10:09:38
5,CNES-ST,"{'uf': 'PE', 'ano': 2012, 'mes': 12}",FTP DATASUS,7337,2026-11-22T10:09:41
6,CNES-ST,"{'uf': 'PE', 'ano': 2013, 'mes': 12}",FTP DATASUS,7930,2026-11-22T10:09:44
7,CNES-ST,"{'uf': 'PE', 'ano': 2014, 'mes': 12}",FTP DATASUS,8364,2026-11-22T10:09:46
8,CNES-ST,"{'uf': 'PE', 'ano': 2015, 'mes': 12}",FTP DATASUS,8368,2026-11-22T10:09:49
9,CNES-ST,"{'uf': 'PE', 'ano': 2016, 'mes': 12}",FTP DATASUS,8661,2026-11-22T10:09:52


Name: pysus
Version: 2.12.2
Summary: Tools for dealing with Brazil's Public health data
Home-page: 
Author: Flavio Codeco Coelho
Author-email: fccoelho@gmail.com
License: GPL
Location: /usr/local/lib/python3.13/dist-packages
Requires: aioftp, anyio, bigtree, boto3, chardet, dateparser, dbfread, dotenv, duckdb, duckdb-engine, fastparquet, httpx, humanize, loguru, numpy, pandas, pyarrow, pydantic, pyreaddbc, python-dateutil, PyYAML, sqlalchemy, tqdm, typer, typing-extensions, Unidecode
Required-by: 
---
Name: differences
Version: 0.3.0
Summary: difference-in-differences estimation and inference in Python
Home-page: https://github.com/bernardodionisi/differences
Author: 
Author-email: Bernardo Dionisi <bernardo.dionisi@gmail.com>
License: GPLv3
Location: /usr/local/lib/python3.13/dist-packages
Requires: formulaic, joblib, numpy, pandas, polars, scipy, statsmodels, tqdm, typing_extensions
Required-by: 
---
Name: linearmodels
Version: 7.1
Summary: Linear Panel, Instrumental Variable, Asset 

Anote as versões e as datas de acesso no `README.md` e no registro de uso de IA.